In [ ]:
# Montar Drive e importar librerías 

from google.colab import drive
drive.mount('/content/drive')

import os
import cv2
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from collections import defaultdict

print("✅ Drive montado y librerías importadas")

In [ ]:
# Rutas del proyecto


BASE       = '/content/drive/MyDrive/mechdog_proyecto'
VIDEOS_DIR = f'{BASE}/videos_raw'
FRAMES_DIR = f'{BASE}/frames'

os.makedirs(VIDEOS_DIR, exist_ok=True)
os.makedirs(FRAMES_DIR, exist_ok=True)

print(f"📁 Videos  → {VIDEOS_DIR}")
print(f"📁 Frames  → {FRAMES_DIR}")

In [ ]:

# Mapa de clases basado en nombres de videos

MAPA_CLASES = {
    '1mochilabien_1sillamal_2mochilabien': 'mochila_ok + silla_riesgo',
    '1mochilamal_2mochilasbien':           'mochila_riesgo + mochila_ok',
    '1mochilabien_1mochilamal':            'mochila_ok + mochila_riesgo',
    '1sillamal_1mochilabien_1cablebien':   'silla_riesgo + mochila_ok + cables_ok',
    '1mochilabien_1cajabien_1cablebien':   'mochila_ok + caja_ok + cables_ok',
    '1mochila1bolsabien':                  'mochila_ok + bolsa_ok',
    '2mochilasmal':                        'mochila_riesgo',
    '2mochilasbien':                       'mochila_ok',
    '2mochilabien':                        'mochila_ok',
    '1mochilamal':  'mochila_riesgo',
    '1mochilabien': 'mochila_ok',
    'mochilamal':   'mochila_riesgo',
    'mochilabien':  'mochila_ok',
    'sillamal':     'silla_riesgo',
    'mesamal':      'mesa_riesgo',
    'cablesbien':   'cables_ok',
    'cablebien':    'cables_ok',
    'cajabien':     'caja_ok',
    'pasillolibre': 'fondo_libre',
}

def detectar_clase(nombre_video):
    nombre_lower = nombre_video.lower()
    for clave, clase in MAPA_CLASES.items():
        if clave in nombre_lower:
            return clase
    return '❓ sin clasificar'

print("✅ Mapa de clases listo —", len(MAPA_CLASES), "entradas")

In [ ]:
# Verificar videos disponibles

extensiones = ('.mp4', '.mov', '.MP4', '.MOV')

videos = sorted([
    f for f in os.listdir(VIDEOS_DIR)
    if any(f.lower().endswith(e.lower()) for e in extensiones)
])

print(f"✅ {len(videos)} videos encontrados:\n")
print(f"  {'Archivo':<58} {'Dur':>6}  {'~Imgs':>6}  Clase detectada")
print(f"  {'─'*100}")

total_dur      = 0
total_imgs_est = 0
sin_clase      = []
doble_ext      = []

for v in videos:
    if v.count('.MP4') + v.count('.mp4') + v.count('.mov') > 1:
        doble_ext.append(v)

    ruta = f'{VIDEOS_DIR}/{v}'
    cap  = cv2.VideoCapture(ruta)
    fps  = cap.get(cv2.CAP_PROP_FPS)
    tot  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    dur  = tot / fps if fps > 0 else 0
    cap.release()

    imgs_est = int(tot / 20)
    clase    = detectar_clase(v)
    total_dur      += dur
    total_imgs_est += imgs_est

    if '❓' in clase:
        sin_clase.append(v); icono = "❓"
    elif 'fondo_libre' in clase: icono = "⬜"
    elif 'riesgo' in clase:      icono = "🔴"
    else:                         icono = "🟢"

    print(f"  {icono} {v:<56} {dur:>5.1f}s  ~{imgs_est:>4}   {clase}")

print(f"  {'─'*100}")
print(f"  {'TOTAL':<58} {total_dur:>5.0f}s  ~{total_imgs_est}")
print(f"\n  Duración total          : {total_dur/60:.1f} minutos")
print(f"  Frames a generar (N=20) : ~{total_imgs_est}")

if doble_ext:
    print(f"\n  ⚠️  DOBLE EXTENSIÓN — renombra antes de continuar:")
    for f in doble_ext: print(f"     → {f}")
if sin_clase:
    print(f"\n  ❓ Sin clase detectada:")
    for f in sin_clase: print(f"     → {f}")
if not doble_ext and not sin_clase:
    print("\n  ✅ Todos los videos están listos para extracción")

In [ ]:

def extraer_frames(video_path, output_dir, cada_n=20):
    os.makedirs(output_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        return 0, 0

    fps_real     = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duracion     = total_frames / fps_real if fps_real > 0 else 0

    frame_idx = 0
    guardados = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % cada_n == 0:
            ruta_out = os.path.join(output_dir, f"frame_{frame_idx:06d}.jpg")
            cv2.imwrite(ruta_out, frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
            guardados += 1
        frame_idx += 1

    cap.release()
    return guardados, duracion

print("Función completada")

In [ ]:
# Extraer todos los frames

CADA_N = 20

print(f"Extrayendo 1 frame cada {CADA_N} fotogramas...\n")
print(f"  {'Video':<58} {'Dur':>6}  {'Imgs':>5}  Clase")
print(f"  {'─'*100}")

resumen   = []
total_img = 0
errores   = []

for video in videos:
    ruta_video = f'{VIDEOS_DIR}/{video}'

    # Limpiar el nombre de carpeta eliminando extensiones intermedias
    nombre_base = video
    for ext in ['.MP4', '.mp4', '.MOV', '.mov']:
        nombre_base = nombre_base.replace(ext, '')
    nombre_base = nombre_base.strip('._- ')

    carpeta_out = f'{FRAMES_DIR}/{nombre_base}'
    clase       = detectar_clase(video)

    n, dur = extraer_frames(ruta_video, carpeta_out, CADA_N)

    if n == 0:
        errores.append(video)
        print(f"  ❌ {video:<56}  ERROR")
        continue

    total_img += n
    icono = "✅" if n >= 40 else "⚠️ "
    print(f"  {icono} {video:<56} {dur:>5.1f}s  {n:>5}   {clase}")

    resumen.append({
        'video':   video,
        'frames':  n,
        'clase':   clase,
        'carpeta': carpeta_out
    })

print(f"  {'─'*100}")
print(f"\n RESULTADO")
print(f"  Videos OK   : {len(resumen)}")
print(f"  Errores     : {len(errores)}")
print(f"  Frames total: {total_img}")
print(f"  Promedio    : {total_img // max(len(resumen),1)} imgs/video")

if errores:
    print(f"\n  ❌ Con error (posible doble extensión):")
    for e in errores:
        print(f"     → {e}")

In [ ]:
# Balance de clases generado

conteo = defaultdict(int)
for item in resumen:
    conteo[item['clase']] += item['frames']

print("Frames generados por clase:\n")
print(f"  {'Clase':<35} {'Frames':>7}  {'%':>6}")
print(f"  {'─'*52}")

for clase, n in sorted(conteo.items(), key=lambda x: -x[1]):
    pct   = n / total_img * 100
    barra = '█' * int(pct / 3)
    nota  = "  (sin etiquetar)" if 'fondo' in clase else ""
    print(f"  {clase:<35} {n:>6}   {pct:>5.1f}%  {barra}{nota}")

print(f"\n  Total frames : {total_img}")
print(f"  Meta Roboflow: seleccionar ~40-50 por video")